# Medical AI Assistant Fine Tuning with QLoRA and Unsloth

**Model:** `unsloth/Llama-3.2-3B-Instruct`  
**Method:** 4-bit Quantisation (QLoRA) + LoRA Adapters  
**Dataset:** `medalpaca/medical_meadow_medqa` (USMLE-style Q&A)  
**Hardware:** Google Colab Free Tier - T4 GPU (15 GB VRAM)  

---
**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## Cell 1 - GPU Check

In [ ]:
# Verify GPU is available before doing anything else
import subprocess, sys, os

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode != 0 or not result.stdout.strip():
    raise RuntimeError(
        'No GPU detected!\n'
        'Fix: Runtime → Change runtime type → T4 GPU → Save → Reconnect'
    )
print('GPU detected:', result.stdout.strip())
print('Python:', sys.version.split()[0])

## Cell 2 - Install Dependencies

In [ ]:
# This cell installs everything needed.
# It takes 3-5 minutes on first run - wait for the message before continuing.
# If Colab shows a 'Restart Runtime' button after this - click it, then re-run from Cell 1.

import subprocess, sys

print('Step 1/2 - Installing Unsloth...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-deps',
    'unsloth @ git+https://github.com/unslothai/unsloth.git'
], check=True)

print('Step 2/2 - Installing HuggingFace / TRL stack...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0',
    'datasets>=2.18.0',
    'peft>=0.13.0',
    'trl>=0.12.0',
    'accelerate>=0.34.0',
    'bitsandbytes>=0.44.0',
    'xformers',
    'sentencepiece',
    'protobuf',
    'psutil',
], check=True)

print('All packages installed!')
print('→ If you see a RESTART button above, click it then re-run from Cell 1.')

## Cell 3 - Imports & Global Config

In [ ]:
import os, gc, time, warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import matplotlib.pyplot as plt

# HuggingFace
from transformers import TrainingArguments
from datasets import load_dataset

# Unsloth - must be imported AFTER transformers
from unsloth import FastLanguageModel, is_bfloat16_supported

# TRL trainer
from trl import SFTTrainer, SFTConfig

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'Device   : {torch.cuda.get_device_name(0)}')
print(f'bf16 OK  : {is_bfloat16_supported()}  (T4=False, A100=True)')
print()

# All hyperparameters in one place
# To run a quick demo (20 min), set max_samples=500 and num_train_epochs=1
CFG = {
    # Model
    'model_name'      : 'unsloth/Llama-3.2-3B-Instruct',
    'max_seq_length'  : 2048,
    'load_in_4bit'    : True,

    # LoRA
    'lora_r'          : 16,
    'lora_alpha'      : 32,
    'lora_dropout'    : 0.05,

    # Dataset
    'dataset_name'    : 'medalpaca/medical_meadow_medqa',
    'max_samples'     : 500,   # ← lower to 500 for a quick demo
    'val_fraction'    : 0.05,
    'seed'            : 42,

    # Training
    'output_dir'            : '/content/medical_outputs',
    'num_train_epochs'      : 1,
    'per_device_batch_size' : 2,
    'grad_accum_steps'      : 4,
    'learning_rate'         : 2e-4,
    'warmup_steps'          : 50,
    'weight_decay'          : 0.01,
    'logging_steps'         : 25,

    # Save paths
    'adapter_path'    : '/content/medical_lora_adapter',
}

os.makedirs(CFG['output_dir'], exist_ok=True)
os.makedirs(CFG['adapter_path'], exist_ok=True)
print('Config loaded')
print(f'Effective batch size: {CFG["per_device_batch_size"] * CFG["grad_accum_steps"]}')

## Cell 4 - GPU Memory Helper

In [ ]:
def gpu_stats(label='GPU Memory'):
    """Print current VRAM usage."""
    if not torch.cuda.is_available():
        print('No CUDA device'); return
    alloc = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    free  = total - alloc
    print(f'  [{label}]  Allocated: {alloc:.2f} GB  |  Free: {free:.2f} GB  |  Total: {total:.2f} GB')

def free_memory():
    """Release unused GPU cache."""
    gc.collect()
    torch.cuda.empty_cache()

gpu_stats('Baseline (no model loaded)')

## Cell 5 - Load Base Model (4-bit QLoRA)

In [ ]:
# What happens here:
#   - Llama-3.2-3B is downloaded from HuggingFace (~1.7 GB in 4-bit NF4)
#   - Weights are quantised from fp16 (6 GB) → 4-bit (1.7 GB) saving 4+ GB VRAM
#   - This is the 'Q' in QLoRA

print('Loading base model in 4-bit...')
t0 = time.time()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CFG['model_name'],
    max_seq_length = CFG['max_seq_length'],
    dtype          = None,               # auto: fp16 on T4, bf16 on A100
    load_in_4bit   = CFG['load_in_4bit'],
)

print(f'Base model loaded in {time.time()-t0:.1f}s')
gpu_stats('After base model load')

total_params = sum(p.numel() for p in model.parameters())
print(f'   Total parameters: {total_params/1e9:.2f}B')

## Cell 6 - Attach LoRA Adapters

In [ ]:
# What happens here:
#   - Small trainable A and B matrices are added beside frozen 4-bit weights
#   - Only these adapter matrices receive gradients during training
#   - Result: ~0.8% of params trainable instead of 100%

model = FastLanguageModel.get_peft_model(
    model,
    r                          = CFG['lora_r'],
    target_modules             = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha                 = CFG['lora_alpha'],
    lora_dropout               = CFG['lora_dropout'],
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',  # Unsloth's optimised checkpointing
    random_state               = CFG['seed'],
    use_rslora                 = False,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'LoRA adapters attached')
print(f'Trainable params : {trainable/1e6:.2f}M  ({100*trainable/total:.2f}% of {total/1e9:.2f}B)')
gpu_stats('After LoRA attach')

## Cell 7 - Load & Preview Dataset

In [ ]:
print(f'Downloading {CFG["dataset_name"]}...')

raw = load_dataset(
    CFG['dataset_name'],
    split='train',
    trust_remote_code=True,
)

print(f'Dataset loaded: {len(raw)} total samples')
print(f'Columns: {raw.column_names}')
print()

# Show 2 sample records
for i in range(2):
    s = raw[i]
    print(f'--- Sample {i+1} ---')
    print(f'Instruction : {str(s.get("instruction", ""))[:150]}')
    if s.get('input', '').strip():
        print(f'Input       : {str(s["input"])[:100]}')
    print(f'Output      : {str(s.get("output", ""))[:200]}')
    print()

## Cell 8 - Format Dataset into Llama-3 Chat Template

In [ ]:
SYSTEM_PROMPT = (
    'You are a knowledgeable and compassionate medical AI assistant. '
    'Provide accurate, evidence-based answers to medical questions. '
    'Always recommend consulting a qualified healthcare professional for personal medical decisions.'
)

def format_sample(sample):
    """Convert a dataset row into the Llama-3 chat format."""
    instruction = str(sample.get('instruction', '')).strip()
    context     = str(sample.get('input', '')).strip()
    answer      = str(sample.get('output', '')).strip()

    user_msg = f'{instruction}\n\nContext:\n{context}' if context else instruction

    # Llama-3 special tokens
    text = (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n'
        f'{SYSTEM_PROMPT}\n'
        '<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n'
        f'{user_msg}\n'
        '<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n'
        f'{answer}\n'
        '<|eot_id|>'
    )
    return {'text': text}

def is_valid(sample):
    """Keep only samples with meaningful instruction and output."""
    inst = str(sample.get('instruction', ''))
    out  = str(sample.get('output', ''))
    return (
        len(inst.strip()) > 10
        and len(out.strip()) > 20
        and (len(inst) + len(out)) < 5000   # avoid sequences too long
    )

# Subsample → filter → format
n = min(CFG['max_samples'], len(raw))
dataset = raw.shuffle(seed=CFG['seed']).select(range(n))

before = len(dataset)
dataset = dataset.filter(is_valid, desc='Filtering')
print(f'Filtered: {before} → {len(dataset)} samples')

dataset = dataset.map(format_sample, remove_columns=dataset.column_names, desc='Formatting')
print(f'Formatted: {len(dataset)} samples')

# Train / validation split
splits = dataset.train_test_split(test_size=CFG['val_fraction'], seed=CFG['seed'])
train_dataset = splits['train']
eval_dataset  = splits['test']

print(f'Train: {len(train_dataset)}  |  Val: {len(eval_dataset)}')
print()
print('--- Formatted example (first 600 chars) ---')
print(train_dataset[0]['text'][:600])

## Cell 9 - Token Length Distribution Check

In [ ]:
# Check that most samples fit within max_seq_length
# Sample 200 rows for speed
sample_size = min(200, len(train_dataset))
lengths = [
    len(tokenizer(ex['text'], truncation=False)['input_ids'])
    for ex in train_dataset.select(range(sample_size))
]
arr = np.array(lengths)

print(f'Token length stats (over {sample_size} samples):')
print(f'  Mean   : {arr.mean():.0f}')
print(f'  Median : {np.median(arr):.0f}')
print(f'  95th % : {np.percentile(arr, 95):.0f}')
print(f'  Max    : {arr.max()}')
pct_over = 100 * (arr > CFG['max_seq_length']).mean()
print(f'  Over max_seq_length ({CFG["max_seq_length"]}): {pct_over:.1f}%')

# Quick histogram
plt.figure(figsize=(8, 3))
plt.hist(lengths, bins=40, color='steelblue', edgecolor='white')
plt.axvline(CFG['max_seq_length'], color='red', linestyle='--', label=f'max_seq_length={CFG["max_seq_length"]}')
plt.xlabel('Token length'); plt.ylabel('Count')
plt.title('Training sample token lengths'); plt.legend()
plt.tight_layout(); plt.show()

## Cell 10 - Configure SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = eval_dataset,
    args          = SFTConfig(
        # Paths
        output_dir                  = CFG['output_dir'],

        # Sequence
        dataset_text_field          = 'text',
        max_seq_length              = CFG['max_seq_length'],
        dataset_num_proc            = 2,
        packing                     = False,

        # Epochs / steps
        num_train_epochs            = CFG['num_train_epochs'],

        # Batch
        per_device_train_batch_size = CFG['per_device_batch_size'],
        per_device_eval_batch_size  = CFG['per_device_batch_size'],
        gradient_accumulation_steps = CFG['grad_accum_steps'],

        # Optimiser
        optim                       = 'adamw_8bit',
        learning_rate               = CFG['learning_rate'],
        weight_decay                = CFG['weight_decay'],
        lr_scheduler_type           = 'cosine',
        warmup_steps                = CFG['warmup_steps'],
        max_grad_norm               = 1.0,

        # Precision - auto-detected
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),

        # Logging
        logging_steps               = CFG['logging_steps'],
        eval_strategy               = 'steps',
        eval_steps                  = 100,
        save_strategy               = 'steps',
        save_steps                  = 200,
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
        greater_is_better           = False,

        # Misc
        seed                        = CFG['seed'],
        dataloader_num_workers      = 0,
        report_to                   = 'none',
    ),
)

# Estimate training time
steps_per_epoch = len(train_dataset) // (CFG['per_device_batch_size'] * CFG['grad_accum_steps'])
total_steps     = steps_per_epoch * CFG['num_train_epochs']
est_min         = total_steps * 1.5 / 60

print(f'Trainer configured')
print(f'Steps per epoch : {steps_per_epoch}')
print(f'Total steps     : {total_steps}')
print(f'Estimated time  : ~{est_min:.0f} min on free T4')
gpu_stats('Before training')

## Cell 11 - Train the Model

In [ ]:
# This is the main training loop.
# You will see loss values printed every 25 steps.
# Expected: loss starts ~2.0-2.5 and should drop toward ~0.8-1.2
#
# DO NOT interrupt this cell unless you see an error.
# A normal disconnection warning can be ignored - Colab will keep running.

print('Starting fine-tuning...')
print('-' * 60)
t_start = time.time()

train_result = trainer.train()

elapsed = time.time() - t_start
print()
print('=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)
print(f'  Time elapsed : {elapsed/60:.1f} min')
print(f'  Total steps  : {train_result.global_step}')
print(f'  Final loss   : {train_result.training_loss:.4f}')
gpu_stats('After training')

## Cell 12 - Plot Training Loss Curve

In [ ]:
history      = trainer.state.log_history
train_steps  = [e['step'] for e in history if 'loss'      in e and 'eval_loss' not in e]
train_losses = [e['loss'] for e in history if 'loss'      in e and 'eval_loss' not in e]
eval_steps   = [e['step'] for e in history if 'eval_loss' in e]
eval_losses  = [e['eval_loss'] for e in history if 'eval_loss' in e]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_steps,  train_losses, label='Train loss', color='#2196F3', linewidth=2)
if eval_losses:
    ax.plot(eval_steps, eval_losses, label='Val loss',
            color='#FF5722', linestyle='--', linewidth=2, marker='o', markersize=5)
ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training & Validation Loss - QLoRA Fine-Tuning', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/content/training_loss_curve.png', dpi=150)
plt.show()
print('Loss curve saved to /content/training_loss_curve.png')

## Cell 13 - Save LoRA Adapter

In [ ]:
# Saves only the small adapter weights (~50-100 MB), not the full base model.
# You can reload this adapter on top of the base model anytime.

print('Saving LoRA adapter...')
model.save_pretrained(CFG['adapter_path'])
tokenizer.save_pretrained(CFG['adapter_path'])
print(f'Adapter saved to: {CFG["adapter_path"]}')
print()

# Show what was saved
print('Saved files:')
for fname in sorted(os.listdir(CFG['adapter_path'])):
    fpath = os.path.join(CFG['adapter_path'], fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {fname:<45}  {size_mb:.1f} MB')

# Save training metrics
trainer.log_metrics('train', train_result.metrics)
trainer.save_metrics('train', train_result.metrics)

## Cell 14 - Inference with Fine-Tuned Model

In [ ]:
# We reuse the already-loaded model (still in memory from training)
# and switch it to inference mode - no need to reload from disk.
# This avoids the OOM error that occurs when loading two models at once on a free T4.

free_memory()
FastLanguageModel.for_inference(model)   # Unsloth 2x speed patch for inference

SYSTEM_PROMPT_INF = SYSTEM_PROMPT  # defined in Cell 8

def build_prompt(question):
    """Build inference prompt - no answer token included."""
    return (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n'
        f'{SYSTEM_PROMPT_INF}\n'
        '<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n'
        f'{question.strip()}\n'
        '<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n'
    )

@torch.inference_mode()
def generate(question, max_new_tokens=350):
    prompt    = build_prompt(question)
    inputs    = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]
    outputs   = model.generate(
        **inputs,
        max_new_tokens     = max_new_tokens,
        temperature        = 0.3,
        top_p              = 0.9,
        do_sample          = True,
        repetition_penalty = 1.15,
        pad_token_id       = tokenizer.eos_token_id,
    )
    new_tokens = outputs[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

TEST_QUESTIONS = [
    'What is the mechanism of action of metformin in treating type 2 diabetes?',
    'A 45-year-old man presents with severe crushing chest pain radiating to the left arm '
    'and diaphoresis. What is the most likely diagnosis and immediate management?',
    'Explain the difference between Type I and Type II hypersensitivity reactions with examples.',
]

print('FINE-TUNED MODEL INFERENCE RESULTS')
print('=' * 70)
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f'\nQ{i}: {q}')
    print('-' * 70)
    answer = generate(q)
    print(answer)
    print()

## Cell 15 - Final Summary

In [ ]:
free_memory()
gpu_stats('Final VRAM snapshot')
print()
print('Pipeline complete!')
print()
print('Saved artefacts:')
print(f'  LoRA adapter   → {CFG["adapter_path"]}')
print(f'  Loss curve     → /content/training_loss_curve.png')
print(f'  Train metrics  → {CFG["output_dir"]}/train_results.json')
print()
print('Next steps:')
print('  • Download adapter from the Files panel on the left (folder icon)')
print('  • Or push to HuggingFace Hub (see optional cell below)')